# Export apex POC-ABS flatten ordered

Standalone notebook. No multiprocessing.

In [ ]:
import os
import re
import sys
from dataclasses import dataclass
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'preprocess-anxiety':
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f'Project root: {ROOT}')


In [ ]:
from comparasion.core.config import ComparisonConfig
from comparasion.core.roi import ROIExtractor
from features_extraction.poc import POC
from features_extraction.quadran import Quadran
from features_extraction.vektor import Vektor


In [ ]:
DATASET_ROOT = (ROOT / 'output/apex/dataset').resolve()
METADATA_PATH = (ROOT / 'output/apex/metadata.xlsx').resolve()
OUTPUT_DIR = (ROOT / 'output/apex/features').resolve()
OUTPUT_PATH = OUTPUT_DIR / 'poc_abs_flatten_ordered_v2.xlsx'

REGIONS = {
    'mulut': list(range(48, 68)),
    'mata_kiri': list(range(17, 22)) + list(range(36, 42)),
    'mata_kanan': list(range(22, 27)) + list(range(42, 48)),
}

TARGET_SIZE = {
    'mulut': (70, 35),
    'mata_kiri': (48, 32),
    'mata_kanan': (48, 32),
}

config = ComparisonConfig(
    output_root=ROOT / 'output/apex/tmp_compare',
    regions=REGIONS,
    target_size=TARGET_SIZE,
)

print(f'Dataset root: {DATASET_ROOT}')
print(f'Metadata:     {METADATA_PATH}')
print(f'Output xlsx:  {OUTPUT_PATH}')
print(f'Regions:      {list(config.regions.keys())}')
print(f'Target size:  {config.target_size}')
print(f'Block size:   {config.block_size}')


In [ ]:
if not DATASET_ROOT.exists():
    raise FileNotFoundError(f'Dataset root not found: {DATASET_ROOT}')
if not METADATA_PATH.exists():
    raise FileNotFoundError(f'Metadata not found: {METADATA_PATH}')
if not config.predictor_path.exists():
    raise FileNotFoundError(f'Predictor not found: {config.predictor_path}')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
metadata = pd.read_excel(METADATA_PATH)
extractor = ROIExtractor(config)

print(f'Metadata rows: {len(metadata)}')
metadata.head(2)


In [ ]:
@dataclass(slots=True)
class EventClip:
    phase: str
    condition: str
    label: str
    participant_raw: str
    participant_key: str
    question: str
    question_no: int
    sample_name: str
    clip_name: str
    event_clip: str
    event_no: int
    event_dir: Path
    onset_frame: int
    apex_frame: int
    offset_frame: int

    @property
    def sample_id(self) -> str:
        return f'{self.phase}__{self.condition}__{self.participant_raw}__{self.question}__{self.sample_name}__event_{self.event_no:03d}'


def natural_sort_key(value: str):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', value)]


def parse_question_no(question: str) -> int:
    match = re.search(r'(\d+)', question)
    return int(match.group(1)) if match else 0


def normalize_participant_key(name: str) -> str:
    return re.sub(r'_[0-9]+$', '', name)


def map_label(condition: str) -> str:
    mapping = {
        'anxiety': 'anxiety_tinggi',
        'tidak': 'anxiety_rendah',
    }
    return mapping.get(condition, condition)


def parse_event_dir(clip_dir_value: str) -> Path:
    rel = str(clip_dir_value).replace('\\', '/')
    if rel.startswith('dataset/'):
        rel = rel[len('dataset/'):]
    return (DATASET_ROOT / Path(rel)).resolve()


def build_clip_path_value(event_dir: Path) -> str:
    try:
        return str(event_dir.resolve().relative_to(ROOT))
    except ValueError:
        return str(event_dir.resolve())


def build_event_clips(df: pd.DataFrame) -> list[EventClip]:
    clips: list[EventClip] = []
    valid = df[df['clip_dir'].notna()].copy()

    for row in valid.itertuples(index=False):
        parts = Path(row.relative_sample).parts
        if len(parts) < 5:
            continue

        phase, condition, participant_raw, question, sample_name = parts[:5]
        event_dir = parse_event_dir(row.clip_dir)
        clips.append(EventClip(
            phase=phase,
            condition=condition,
            label=map_label(condition),
            participant_raw=participant_raw,
            participant_key=normalize_participant_key(participant_raw),
            question=question,
            question_no=parse_question_no(question),
            sample_name=sample_name,
            clip_name=sample_name,
            event_clip=event_dir.name,
            event_no=int(row.event_no),
            event_dir=event_dir,
            onset_frame=int(row.onset_frame),
            apex_frame=int(row.apex_frame),
            offset_frame=int(row.offset_frame),
        ))

    return sorted(
        clips,
        key=lambda c: (
            c.participant_key,
            c.label,
            c.question_no,
            0 if c.phase == 'before' else 1,
            natural_sort_key(c.clip_name),
            c.event_no,
            natural_sort_key(c.event_clip),
        ),
    )


event_clips = build_event_clips(metadata)
print(f'Event clips: {len(event_clips)}')
event_clips[:3]


In [ ]:
def load_event_frames(event_dir: Path) -> tuple[list[np.ndarray], list[Path]]:
    frame_paths = sorted(
        [p for p in event_dir.iterdir() if p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg', '.png'}],
        key=lambda p: natural_sort_key(p.name),
    )
    if len(frame_paths) < 2:
        raise ValueError(f'Need >= 2 frames: {event_dir}')

    frames = []
    for path in frame_paths:
        image = cv2.imread(str(path))
        if image is None:
            raise FileNotFoundError(f'Failed to read image: {path}')
        frames.append(image)
    return frames, frame_paths


def to_gray(image: np.ndarray) -> np.ndarray:
    if image.ndim == 2:
        return image
    return cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)


def extract_flatten_row(clip: EventClip, frame_no: int, rois: dict[str, np.ndarray], baseline_gray: dict[str, np.ndarray], block_size: int) -> dict:
    row = {
        'phase': clip.phase,
        'condition': clip.condition,
        'label': clip.label,
        'participant': clip.participant_key,
        'participant_raw': clip.participant_raw,
        'question': clip.question,
        'question_no': clip.question_no,
        'sample': clip.sample_name,
        'clip': clip.clip_name,
        'event_clip': clip.event_clip,
        'event_no': clip.event_no,
        'clip_path': build_clip_path_value(clip.event_dir),
        'frame': frame_no,
    }

    for comp in config.regions:
        gray = to_gray(rois[comp])
        poc = POC(baseline_gray[comp], gray, block_size)
        vec = Vektor(poc.getPOC(), block_size)
        quad = Quadran(vec.getVektor()).getQuadran()

        for block_id, block in enumerate(quad, start=1):
            row[f'{comp}_x{block_id}'] = block[1]
            row[f'{comp}_y{block_id}'] = block[2]
            row[f'{comp}_t{block_id}'] = block[3]
            row[f'{comp}_m{block_id}'] = block[4]

    return row


def process_event_clip(clip: EventClip) -> list[dict]:
    frames, _frame_paths = load_event_frames(clip.event_dir)

    rois_per_frame: list[dict[str, np.ndarray] | None] = []
    for frame in frames:
        try:
            rois_per_frame.append(extractor.extract_rois(frame))
        except ValueError:
            rois_per_frame.append(None)

    baseline_rois = rois_per_frame[0]
    if baseline_rois is None:
        raise ValueError(f'No face detected in baseline frame: {clip.event_dir}')

    baseline_gray = {name: to_gray(img) for name, img in baseline_rois.items()}
    rows: list[dict] = []

    for idx in range(1, len(frames)):
        rois = rois_per_frame[idx]
        if rois is None:
            continue
        rows.append(extract_flatten_row(clip, idx + 1, rois, baseline_gray, config.block_size))

    return rows


In [ ]:
all_rows = []
errors = []

for idx, clip in enumerate(event_clips, start=1):
    print(f'[{idx}/{len(event_clips)}] {clip.phase}/{clip.label}/{clip.participant_key}/{clip.question}/{clip.clip_name}/{clip.event_clip}/event_{clip.event_no:03d}')
    try:
        rows = process_event_clip(clip)
        all_rows.extend(rows)
    except Exception as exc:
        errors.append((clip.sample_id, str(exc)))

print(f'Rows collected: {len(all_rows)}')
print(f'Errors: {len(errors)}')


In [ ]:
if errors:
    error_df = pd.DataFrame(errors, columns=['sample_id', 'error'])
    print(error_df.head(20).to_string(index=False))
else:
    error_df = pd.DataFrame(columns=['sample_id', 'error'])

error_df.head(20)


In [ ]:
df = pd.DataFrame(all_rows)
if df.empty:
    raise ValueError('No rows produced.')

df = df.sort_values(
    by=['participant', 'label', 'question_no', 'phase', 'clip', 'event_clip', 'event_no', 'frame'],
    key=lambda s: s.map({'before': 0, 'after': 1}) if s.name == 'phase' else s,
    kind='stable',
).reset_index(drop=True)

df.to_excel(OUTPUT_PATH, index=False)
print(f'Saved: {OUTPUT_PATH}')
print(f'Shape: {df.shape}')
df.head()


In [ ]:
print(df[['participant', 'label', 'question', 'phase', 'clip', 'event_clip', 'event_no', 'frame']].head(20).to_string(index=False))
print()
print(df['phase'].value_counts().to_dict())
print(df['label'].value_counts().to_dict())
